In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

# Pooling Layers

The _pooling_ operation helps reduce the spatial dimensionality of the feature maps, and therefore the number of parameters of the network. Pooling layers are basically convolutions in which the stride is usually equal to the kernel size and in which some function is computed over all the pixels. The most common choices are the maximum, the minimum or the average. They not only reduce dimensionality (and thus make training faster), but they also generally help with classification, since they enable more effective generalization.

A common practice when building a convolutional architecture is to interleave ``Conv2d`` layers with ``Pooling`` layers until reaching ``Dense`` (Feed-Forward) layers that will discriminate the features learned by the previous layers.


## Max Pool

### Forward Method

The _Max Pooling_ operation is similar to convolution, but it does not require an element-wise weighted sum, since in this case we select the maximum element of each particular region enclosed by the kernel.

Its computation is easy to explain with an example: suppose $x \in \mathbb{R}^{(4\times 4)}$ and $F=S=2$, so that the $y_{ij}$ are defined as:

$$
\begin{array}{cc}
y_{11} = \max ( x_{11},x_{12},x_{21},x_{22} ) & y_{12} = \max ( x_{13},x_{14},x_{23},x_{24} ) \\
y_{21} = \max ( x_{31},x_{32},x_{41},x_{42} ) & y_{22} = \max ( x_{33},x_{34},x_{43},x_{44} )
\end{array}
$$

This lets us see that the output shape of a Max Pool 2D layer is described by the same equation as the output shape of a convolutional layer.

Implement the `forward` method of the `MaxPool2d` model in the file `edunn/models/pooling.py`.

In [ ]:
np.random.seed(123)

x = np.random.rand(2,3,7,7)

kernel_size,stride=2,2

layer=nn.MaxPool2d(kernel_size=kernel_size, stride=stride)

y=layer.forward(x)
y, y.shape

### Backward Method

Knowing the gradient of the error function with respect to the output pixels, we must compute the total gradient of the error with respect to the input pixels. Using the chain rule and partial derivatives we arrive at the expression:

$$
\frac{\partial E}{\partial x_{mn}}=\sum_{ij}\frac{\partial E}{\partial y_{ij}}\frac{\partial y_{ij}}{\partial x_{mn}} 
$$

Note that pooling layers have no parameters, so there is no kernel gradient to compute.


#### `δE/δx`

The gradients of the error $E$ with respect to the input feature map $x$ can be obtained by taking partial derivatives, as explained in previous guides.

Continuing with the previous example, it is clear that $\frac{\partial y_{ij}}{\partial x_{mn}}$ is nonzero only if $x_{mn}$ is the maximum element of the region of $y_{ij}$. So if $x_{12}$ happens to be the maximum element of the region covered by $y_{11}$, then:

$$\frac{\partial y_{11}}{\partial x_{12}}=\frac{\partial x_{12}}{\partial x_{12}}=1$$

And the remaining derivatives with respect to the other $x_{mn}$ of the first region will be zero. The derivative of $E$ with respect to $x_{12}$ is obtained by multiplying the local gradient by the gradients coming from the next layer, so that only $\frac{\partial E}{\partial x_{12}}=\frac{\partial E}{\partial y_{11}} \cdot 1 \neq 0$ remains for the region of $y_{11}$.

* The same reasoning holds for all  $(i,j)$  regions.

![maxpool_backward.png](img/maxpool_backward.png)

In [ ]:
# Define the gradient of the output
g = np.random.rand(*y.shape)

# Propagate the gradient backward through the convolution
layer_grad = layer.backward(g)
layer_grad

### Checks

In [ ]:
x = np.array([[[[1,5,2,0],
                [3,4,8,1],
                [0,2,7,3],
                [6,1,4,9]]]])

layer=nn.MaxPool2d(kernel_size=2, stride=2)
y = np.array([[[[5,8],
                [6,9]]]])
utils.check_same(y,layer.forward(x))

In [ ]:
np.random.seed(123)

samples = 100
batch_size=2
din=3 # input dimension
dout=5 # output dimension
input_shape=(batch_size,din,11,11)

# Check the derivatives of a Flatten model
# with random values for `w`, `b`, and `x`, the input
layer=nn.MaxPool2d(kernel_size=3)


utils.check_gradient.common_layer(layer,input_shape,samples=samples,tolerance=1e-5)    

## Avg Pool

### Forward Method

The _Average Pooling_ operation is similar to convolution, where we compute the average of the elements that make up the particular region enclosed by the kernel.

Its computation is easy to explain with an example: suppose $x \in \mathbb{R}^{(4\times 4)}$, $F=2$ and $S=1$, so that the $y_{ij}$ are defined as:

$$
\begin{array}{ccc}
y_{11} = \text{avg} ( x_{11},x_{12},x_{21},x_{22} ) & 
y_{12} = \text{avg} ( x_{12},x_{13},x_{22},x_{23} ) & 
y_{13} = \text{avg} ( x_{13},x_{14},x_{23},x_{24} ) \\

y_{21} = \text{avg} ( x_{21},x_{22},x_{31},x_{32} ) & 
y_{22} = \text{avg} ( x_{22},x_{23},x_{32},x_{33} ) & 
y_{23} = \text{avg} ( x_{23},x_{24},x_{33},x_{34} ) \\

y_{31} = \text{avg} ( x_{31},x_{32},x_{41},x_{42} ) & 
y_{32} = \text{avg} ( x_{32},x_{33},x_{42},x_{43} ) & 
y_{33} = \text{avg} ( x_{33},x_{34},x_{43},x_{44} )
\end{array}
$$

This lets us see that the output shape of an Avg Pool 2D layer is described by the same equation as the output shape of a convolutional layer.

Implement the `forward` method of the `AvgPool2d` model in the file `edunn/models/pooling.py`.

> TIP: you can reuse the forward function of the `MaxPool2d` layer and replace `np.max` with `np.average`.

In [ ]:
np.random.seed(123)

x = np.random.rand(2,3,7,7)
# x = np.random.rand(1,1,4,4)

kernel_size,stride=2,1

layer=nn.AvgPool2d(kernel_size=kernel_size, stride=stride)

y=layer.forward(x)
y, y.shape

### Backward Method

#### `δE/δx`

The gradients of the error $E$ with respect to the input feature map $x$ can be obtained by taking partial derivatives, as explained in previous guides.

<!-- Since we have to compute the derivative of one matrix with respect to another, each element of the matrix is treated as an independent variable. Therefore, the derivative of an element of the matrix $y$ with respect to an element of the matrix $x$ takes into account every possible path through which that element of $x$ can influence the element of $y$.

In this way, $\frac{\partial y}{\partial x_{mn}} = \sum_{ij} \frac{\partial y_{ij}}{\partial x_{mn}}$, by the chain rule for partial derivatives, which tells us that the derivative of a function with respect to a variable is the sum of the derivatives of that function along all the possible paths (some of which may be zero terms, because certain $y_{ij}$ do not depend on $x_{mn}$). -->

Continuing with the previous example, it is clear that $\frac{\partial y_{ij}}{\partial x_{mn}}$ is nonzero only if $x_{mn}$ belongs to the region of $y_{ij}$, in which case it also equals $\frac{1}{F_H \cdot F_W}$. So, to compute $\frac{\partial y}{\partial x_{12}}$ for example:

$$\frac{\partial y}{\partial x_{12}} = \frac{\partial y_{11}}{\partial x_{12}} + \frac{\partial y_{12}}{\partial x_{12}} + \frac{\partial y_{21}}{\partial x_{12}} + \frac{\partial y_{22}}{\partial x_{12}} = \frac{1}{4} + \frac{1}{4} + 0 + 0$$

since:

$$\begin{aligned}
y_{11} &= \frac { x_{11}+\color{red}{x_{12}}\color{default}+x_{21}+x_{22} }{F_H \cdot F_W} \Rightarrow 
    \frac{\partial y_{11}}{\partial x_{12}} = \frac{1}{4} \frac{\partial x_{12}}{\partial x_{12}} = \frac{1}{4} \\
y_{12} &= \frac { \color{red}{x_{12}}\color{default}+x_{13}+x_{22}+x_{23} }{F_H \cdot F_W} \Rightarrow
    \frac{\partial y_{12}}{\partial x_{12}} = \frac{1}{4} \frac{\partial x_{12}}{\partial x_{12}} = \frac{1}{4} \\
y_{21} &= \frac { x_{21}+x_{22}+x_{31}+x_{32} }{F_H \cdot F_W} \Rightarrow 
    \frac{\partial y_{21}}{\partial x_{12}} = 0 \\
y_{22} &= \frac { x_{22}+x_{23}+x_{32}+x_{33} }{F_H \cdot F_W} \Rightarrow 
    \frac{\partial y_{22}}{\partial x_{12}} = 0
\end{aligned}$$


In this way, the remaining derivatives with respect to the other $x_{mn}$ of the first region will **start out** equal to $\frac{1}{F_H \cdot F_W}$, and will keep growing by that same amount according to how much they influence the output, which depends on the value of $S$. 

This is easier to visualize when $S=2$: there it is clear that, for the first region, the partial result of $\frac{\partial y}{\partial x_{mn}}$ for $m,n \in \{1,2\}$, given that the regions defined by the kernel do not depend on each other, is:

$$
\begin{bmatrix}
1/4 & 1/4 & 0 & 0 \\
1/4 & 1/4 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

* The same reasoning holds for all  $(i,j)$  regions.


In [ ]:
np.set_printoptions(formatter={'float': lambda x: "{0:0.4f}".format(x)})

In [ ]:
# Define the gradient of the output
g = np.random.rand(*y.shape)
# g = np.ones_like(y)

# Propagate the gradient backward through the convolution
layer_grad = layer.backward(g)
layer_grad

### Checks

In [ ]:
x = np.array([[[[1,5,2,0],
                [3,4,8,1],
                [0,2,7,3],
                [6,1,4,9]]]])

layer=nn.AvgPool2d(kernel_size=2, stride=2)
y = np.array([[[[3.25,2.75],
                [2.25,5.75]]]])
utils.check_same(y,layer.forward(x))

In [ ]:
samples = 100
batch_size=2
din=3 # input dimension
dout=5 # output dimension
input_shape=(batch_size,din,11,11)

# Check the derivatives of a Flatten model
# with random values for `w`, `b`, and `x`, the input
layer=nn.AvgPool2d(kernel_size=3, stride=3)


utils.check_gradient.common_layer(layer,input_shape,samples=samples,tolerance=1e-5)    